Weather Data for SEIS 763 Group project by Andrew Miller  
Location for Avila Adobe:  
    Latitude: 34.0572  
    Longitude: -118.2378  
    Elevation 89  


#1 Import Libraries and Set Up Notebook

In [72]:
# Importing necessary libraries
import pandas as pd
from pathlib import Path
from datetime import date 
import meteostat as ms


#2 Create Folder Paths

In [73]:
# Define file paths
RAW_DIR = Path("../data/raw/weather")
PROCESSED_DIR = Path("../data/processed")

# Create directories if they don't exist
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Define file paths for raw and processed data
raw_file = RAW_DIR / "weather_raw.csv"
processed_file = PROCESSED_DIR / "weather_monthly.csv"


#3 Location, Date Range, and Nearby Station

In [74]:
# Define the location for Avila Adobe in Los Angeles (latitude, longitude, elevation)
la_point = ms.Point(34.0572, -118.2437, 89)

# Define the time period for data retrieval
start = date(2014, 1, 1)
end = date(2021, 6, 1)

# Find nearby weather stations
stations = ms.stations.nearby(la_point, limit=4)

#4 Build Daily Time Series and Interpolate Exact Point

In [75]:
# Fetch and interpolate weather data for the specified time period and location
ts = ms.daily(stations, start, end)

weather_raw = ms.interpolate(ts, la_point).fetch()

print(type(weather_raw))
print(weather_raw.head())

<class 'pandas.core.frame.DataFrame'>
            temp  tmin  tmax  rhum  prcp  snwd  wspd    pres  cldc
time                                                              
2014-01-01  13.8   7.5  20.2    64   0.1     0   3.7  1018.7     0
2014-01-02  16.6   9.8  26.4    42   0.0     0   4.1  1017.2     0
2014-01-03  14.4   8.7  20.0    68   0.0     0   3.9  1014.9     0
2014-01-04  14.3   9.7  20.4    71   0.0     0   4.0  1014.8     0
2014-01-05  15.3   8.2  25.0    44   0.1     0   3.5  1018.8     0


In [76]:
# Reset index and rename columns for better readability
weather_raw = weather_raw.reset_index().rename(columns={"time": "date"})
weather_raw.head()

,date,temp,tmin,tmax,rhum,prcp,snwd,wspd,pres,cldc
0,2014-01-01,13.8,7.5,20.2,64,0.1,0,3.7,1018.7,0
1,2014-01-02,16.6,9.8,26.4,42,0.0,0,4.1,1017.2,0
2,2014-01-03,14.4,8.7,20.0,68,0.0,0,3.9,1014.9,0
3,2014-01-04,14.3,9.7,20.4,71,0.0,0,4.0,1014.8,0
4,2014-01-05,15.3,8.2,25.0,44,0.1,0,3.5,1018.8,0


#5 Save the Raw Weather Dataset

In [77]:
# Save raw weather data to CSV
weather_raw.to_csv(raw_file, index=False)
print(f"Raw weather data saved to {raw_file}")

Raw weather data saved to ../data/raw/weather/weather_raw.csv


#6 Cleaning the Weather Dataset

In [78]:
# Display column names for reference
weather_raw.columns.tolist()

['date',
 'temp',
 'tmin',
 'tmax',
 'rhum',
 'prcp',
 'snwd',
 'wspd',
 'pres',
 'cldc']

In [79]:
# Create a copy of the raw weather data for processing
weather = weather_raw.copy()
weather["date"] = pd.to_datetime(weather["date"])
weather.head()

,date,temp,tmin,tmax,rhum,prcp,snwd,wspd,pres,cldc
0,2014-01-01,13.8,7.5,20.2,64,0.1,0,3.7,1018.7,0
1,2014-01-02,16.6,9.8,26.4,42,0.0,0,4.1,1017.2,0
2,2014-01-03,14.4,8.7,20.0,68,0.0,0,3.9,1014.9,0
3,2014-01-04,14.3,9.7,20.4,71,0.0,0,4.0,1014.8,0
4,2014-01-05,15.3,8.2,25.0,44,0.1,0,3.5,1018.8,0


In [80]:
# Rename columns to more descriptive names
weather = weather.rename(columns={
    "tavg": "avg_temp_daily",
    "temp": "avg_temp_daily",
    "tmin": "min_temp_daily",
    "tmax": "max_temp_daily",
    "prcp": "precip_daily",
    "wspd": "wind_daily"
})
weather.head()

,date,avg_temp_daily,min_temp_daily,max_temp_daily,rhum,precip_daily,snwd,wind_daily,pres,cldc
0,2014-01-01,13.8,7.5,20.2,64,0.1,0,3.7,1018.7,0
1,2014-01-02,16.6,9.8,26.4,42,0.0,0,4.1,1017.2,0
2,2014-01-03,14.4,8.7,20.0,68,0.0,0,3.9,1014.9,0
3,2014-01-04,14.3,9.7,20.4,71,0.0,0,4.0,1014.8,0
4,2014-01-05,15.3,8.2,25.0,44,0.1,0,3.5,1018.8,0


In [81]:
# Select only the relevant columns for analysis
keep_cols = [c for c in [
    "date",
    "avg_temp_daily",
    "min_temp_daily",
    "max_temp_daily",
    "precip_daily",
    "wind_daily"
] if c in weather.columns]

weather = weather[keep_cols].sort_values("date").reset_index(drop=True)
weather.head()

,date,avg_temp_daily,min_temp_daily,max_temp_daily,precip_daily,wind_daily
0,2014-01-01,13.8,7.5,20.2,0.1,3.7
1,2014-01-02,16.6,9.8,26.4,0.0,4.1
2,2014-01-03,14.4,8.7,20.0,0.0,3.9
3,2014-01-04,14.3,9.7,20.4,0.0,4.0
4,2014-01-05,15.3,8.2,25.0,0.1,3.5


In [82]:
# Check for missing values in the weather data
weather.isna().sum()

date              0
avg_temp_daily    0
min_temp_daily    0
max_temp_daily    0
precip_daily      0
wind_daily        0
dtype: int64

#7 Create Monthly Key and Aggregate Into Monthly Data

In [83]:
# Create a month column for monthly aggregation
weather["month"] = weather["date"].dt.strftime("%Y-%m")
weather.head()

,date,avg_temp_daily,min_temp_daily,max_temp_daily,precip_daily,wind_daily,month
0,2014-01-01,13.8,7.5,20.2,0.1,3.7,2014-01
1,2014-01-02,16.6,9.8,26.4,0.0,4.1,2014-01
2,2014-01-03,14.4,8.7,20.0,0.0,3.9,2014-01
3,2014-01-04,14.3,9.7,20.4,0.0,4.0,2014-01
4,2014-01-05,15.3,8.2,25.0,0.1,3.5,2014-01


In [84]:
# Define aggregation functions for monthly summary
agg_dict = {}

if "avg_temp_daily" in weather.columns:
    agg_dict["avg_temp_daily"] = "mean"
if "min_temp_daily" in weather.columns:
    agg_dict["min_temp_daily"] = "mean"
if "max_temp_daily" in weather.columns:
    agg_dict["max_temp_daily"] = "mean"
if "precip_daily" in weather.columns:
    agg_dict["precip_daily"] = "sum"
if "wind_daily" in weather.columns:
    agg_dict["wind_daily"] = "mean"

# Aggregate weather data by month using the defined aggregation functions
weather_monthly = weather.groupby("month", as_index=False).agg(agg_dict)
weather_monthly.head()

,month,avg_temp_daily,min_temp_daily,max_temp_daily,precip_daily,wind_daily
0,2014-01,16.174194,10.429032,23.048387,0.5,4.945161
1,2014-02,15.325,11.2,20.45,86.6,7.521429
2,2014-03,16.603226,12.651613,21.43871,14.8,8.767742
3,2014-04,17.293333,12.913333,22.326667,5.2,10.376667
4,2014-05,20.516129,16.232258,25.596774,0.0,10.645161


In [85]:
# Convert temperature from Celsius to Fahrenheit
for col in ["avg_temp_daily", "min_temp_daily", "max_temp_daily"]:
    if col in weather_monthly.columns:
        weather_monthly[col] = (weather_monthly[col] * 9/5) + 32

# Convert precipitation from mm to inches
if "precip_daily" in weather_monthly.columns:
    weather_monthly["precip_daily"] = weather_monthly["precip_daily"] * 0.0393701

# Convert wind speed from km/h to mph
if "wind_daily" in weather_monthly.columns:
    weather_monthly["wind_daily"] = weather_monthly["wind_daily"] * 0.621371

# Round
weather_monthly = weather_monthly.round(2)

weather_monthly.head()

,month,avg_temp_daily,min_temp_daily,max_temp_daily,precip_daily,wind_daily
0,2014-01,61.11,50.77,73.49,0.02,3.07
1,2014-02,59.58,52.16,68.81,3.41,4.67
2,2014-03,61.89,54.77,70.59,0.58,5.45
3,2014-04,63.13,55.24,72.19,0.2,6.45
4,2014-05,68.93,61.22,78.07,0.0,6.61


#8 Rename Monthly Columns, Check for Duplicate Months and Check Dataset Shape

In [91]:
# Rename columns to monthly summary names for clarity
weather_monthly = weather_monthly.rename(columns={
    "avg_temp_daily": "avg_temp_F",
    "min_temp_daily": "min_temp_F",
    "max_temp_daily": "max_temp_F",
    "precip_daily": "total_precip_in",
    "wind_daily": "avg_wind_mph"
})


weather_monthly.head()

,month,avg_temp_F,min_temp_F,max_temp_F,total_precip_in,avg_wind_mph
0,2014-01,61.11,50.77,73.49,0.02,3.07
1,2014-02,59.58,52.16,68.81,3.41,4.67
2,2014-03,61.89,54.77,70.59,0.58,5.45
3,2014-04,63.13,55.24,72.19,0.2,6.45
4,2014-05,68.93,61.22,78.07,0.0,6.61


In [92]:
# Units for reference
units = {
    "avg_temp_F": "Fahrenheit",
    "min_temp_F": "Fahrenheit",
    "max_temp_F": "Fahrenheit",
    "total_precip_in": "inches",
    "avg_wind_mph": "mph"
}

units

{'avg_temp_F': 'Fahrenheit',
 'min_temp_F': 'Fahrenheit',
 'max_temp_F': 'Fahrenheit',
 'total_precip_in': 'inches',
 'avg_wind_mph': 'mph'}

In [93]:
# Check for duplicate month entries in the monthly summary
weather_monthly["month"].duplicated().sum()

# Display summary information about the monthly weather data
print("Rows:", len(weather_monthly))
print("First month:", weather_monthly["month"].min())
print("Last month:", weather_monthly["month"].max())

Rows: 90
First month: 2014-01
Last month: 2021-06


#9 Save the Processed Weather Dataset

In [94]:
# Save the processed monthly weather data to CSV
weather_monthly.to_csv(processed_file, index=False)
print(f"Saved processed weather file to: {processed_file}")

Saved processed weather file to: ../data/processed/weather_monthly.csv


In [95]:
weather_monthly.head(12)

,month,avg_temp_F,min_temp_F,max_temp_F,total_precip_in,avg_wind_mph
0,2014-01,61.11,50.77,73.49,0.02,3.07
1,2014-02,59.58,52.16,68.81,3.41,4.67
2,2014-03,61.89,54.77,70.59,0.58,5.45
3,2014-04,63.13,55.24,72.19,0.2,6.45
4,2014-05,68.93,61.22,78.07,0.0,6.61
5,2014-06,67.87,62.27,75.75,0.0,6.38
6,2014-07,72.63,66.94,80.95,0.07,6.42
7,2014-08,72.43,66.22,80.96,0.03,6.2
8,2014-09,74.16,67.48,83.14,0.01,5.65
9,2014-10,70.32,62.59,80.19,0.29,4.41


In [96]:
weather_monthly.describe()

,avg_temp_F,min_temp_F,max_temp_F,total_precip_in,avg_wind_mph
count,90.0,90.0,90.0,90.0,90.0
mean,65.099,57.770778,74.331556,0.854778,4.815444
std,5.911566,6.541975,6.071289,1.430116,1.67223
min,52.94,46.19,60.14,0.0,1.76
25%,60.775,51.94,69.5375,0.0,3.235
50%,63.92,56.885,73.865,0.14,4.995
75%,70.2475,62.9,80.0,1.155,6.405
max,76.51,69.88,85.12,6.6,7.21
